In [1]:
%load_ext autoreload
%autoreload 2

import frame.frame_assembler as fa
import frame.frame as fr
from datetime import datetime
from elasticsearch import Elasticsearch


client = Elasticsearch("http://localhost:9200/", api_key="WGRuTF9wUUIzYVpjeXh5Wnl2RlA6Q3ZPUFJoZXRUX1NiX3NWQ0FGbHZEdw==")

In [2]:
'''
Counting
'''

source = False
size = 0
query = { "term": { "process.tag.service@namespace": "hifi_1" } }
aggregations = {
    "count_traces": {
        "filter": { "term": { "operationName": "process_digitiser_trace_message" } }
    },
    "count_digitiser_eventlists": {
        "filter": { "term": { "operationName": "process_digitiser_event_list_message" } }
    },
    "count_frame_eventlists": {
        "filter": { "term": { "operationName": "process_frame_assembled_event_list_message" } }
    },
    "count_Frames": {
        "filter": { "term": { "operationName": "Frame" } }
    },
    "count_incomplete_Frames": {
        "filter": { "bool": { "must": [
            { "term": { "operationName": "Frame" } },
            { "term": { "tag.frame_is_expired": "true" } }
        ] } }
    },
    "count_discarded_digitiser_eventlists": {
        "filter": { "bool": { "must": [
            { "term": { "operationName": "process_digitiser_event_list_message" } },
            { "term": { "tag.is_discarded": "true" } }
        ] } }
    }
}
#result = client.search(index="jaeger-span-2025-02-13-*", query = query, source=source, aggregations=aggregations, size = size)

In [4]:
import time
'''
Extracting Incomplete Frames
'''

frames = fa.FrameAssembler(client, "hifi_1", "2025-02-16", 10000)
#musts = [{ "term": { "tag.frame_is_expired": "true" } }]
musts = [ {
    "function_score": {
        "random_score": {
            "seed": time.time_ns(),
            "field": "tag.metadata_timestamp"
        }
    }
}]
shoulds = []
filters = []
frames.find_frames(musts, shoulds, filters, "2025-02-16-10", 10)

Finding Digitiser Event List
Finding process_digitiser_event_list_message
Finding process_kafka_message (digitiser-aggregator)
Finding process_digitiser_trace_message
Finding process_kafka_message (trace-to-events)
Finding process
Finding process_kafka_message (nexus-writer)


In [7]:
the_frames = [fr.Frame(frame,frames) for frame in frames.frames]
for frame in the_frames:
    frame.print_hierarchy()

 Frame
 -<kafka_timestamp_ms = 1739700856725
 -<metadata_timestamp = 2025-02-16T10:14:16.448526960+00:00
|-- Digitiser Event List
|--|-- Process Digitiser Event List Message
|--|-- -<digitiser_id = 8
|--|--|-- Process Kafka Message
|--|--|-- -<kafka_timestamp_ms = 1739700856719
|--|--|--|-- Process Digitiser Trace Message
|--|--|--|--|-- Process Kafka Message
|--|--|--|--|-- -<kafka_timestamp_ms = 1739700856704
|-- Digitiser Event List
|--|-- Process Digitiser Event List Message
|--|-- -<digitiser_id = 6
|--|--|-- Process Kafka Message
|--|--|-- -<kafka_timestamp_ms = 1739700856720
|--|--|--|-- Process Digitiser Trace Message
|--|--|--|--|-- Process Kafka Message
|--|--|--|--|-- -<kafka_timestamp_ms = 1739700856711
|-- Digitiser Event List
|--|-- Process Digitiser Event List Message
|--|-- -<digitiser_id = 10
|--|--|-- Process Kafka Message
|--|--|-- -<kafka_timestamp_ms = 1739700856720
|--|--|--|-- Process Digitiser Trace Message
|--|--|--|--|-- Process Kafka Message
|--|--|--|--|-- -